# How to generate custom basis spectra
The `pah_spec` package relies on a set of pre-computed basis spectra $\tilde{p}_{\lambda_{\rm em}}(\lambda_{\rm abs})$ (Eq. 3 of [Richie & Hensley 2026](https://ui.adsabs.harvard.edu/abs/2025arXiv251016861R/abstract)) to generate PAH spectra, which requires the specification of a PAH energy and absorption cross-section ($C_{\rm abs}$) model. We provide a default set of pre-computed $\tilde{p}_{\lambda_{\rm em}}(\lambda_{\rm abs})$ using the PAH physics model described in Section 3 of [Richie & Hensley 2026](https://ui.adsabs.harvard.edu/abs/2025arXiv251016861R/abstract) (based on the [Draine et al. 2021](https://ui.adsabs.harvard.edu/abs/2021ApJ...917....3D/abstract) model). 

However, since the single photon approximation (SPA) is agnostic to the choice of PAH physics model, one may wish to employ a different PAH model to generate SPA spectra. This notebook illustrates how to specify your own $E(T)$ and/or $C_{\rm abs}(\lambda)$ to create custom basis spectra, and how to use these basis spectra to generate integrated PAH emission spectra.

In [11]:
import astropy.units as u
import numpy as np
import pah_spec  # import the pah_spec module

First, make sure that the required datasets are downloaded
- these functions exit early if the data was already retrieved
- downloads of some of the files can take a while since they are large (~1 GB). If the [tqdm package](https://tqdm.github.io/) is installed, a progressbar is shown

In [12]:
pah_spec.retrieve_internal_data()
pah_spec.retrieve_sample_basis()

Next, initialize the `PahSpec` object.

In [13]:
ps = pah_spec.PahSpec()

Then, define the emission wavelength ($\lambda_{\rm em}$) array. It must start at at least $0.1~{\rm \mu m}$ and extend out to at least $10^4~{\rm \mu m}$ for the PAH to cool correctly. Below we define the emission wavelengths used in the default $\tilde{p}_{\lambda_{\rm em}}(\lambda_{\rm abs})$, but feel free to specify your own.

In [14]:
lower = 0.1
mid_1 = 1
mid_2 = 20
upper = 1e4

emission_short = np.logspace(np.log10(lower), np.log10(mid_1), 232)  # R = 100
emission_jwst = np.logspace(np.log10(mid_1), np.log10(mid_2), 8090)  # R = 2700
emission_long = np.logspace(np.log10(mid_2), np.log10(upper), 624)  # R = 100

emission_wavelengths = (
    np.concatenate((emission_short[:-1], emission_jwst, emission_long[1:])) * u.um
)

We'll use the `PahSpec.generate_basis_spectra()` method to generate the basis spsectra.

In [15]:
print(ps.generate_basis_spectra.__doc__)

Generates basis spectra file for input grain sizes for an ionized or neutral PAHs.

Parameters
----------
grain_sizes : astropy.units.Quantity (float or array_like)
    Array of dust grain radii to calculate basis spectra for
emission_wavelengths : astropy.units.Quantity (array_like)
    Array of emission wavelengths to define basis spectra over
output_directory : str, optional
    Directory to output basis spectra to, default is ./
ion : Bool, optional
    PAH ionization to run basis spectra for, default is False
lambda_min : astropy.units.Quantity, optional
    Lowest lambda_abs wavelength, recommended default is 912 A
lambda_max : astropy.units.Quantity, optional
    Highest lambda_abs wavelength, recommended default is 10 um

Returns
-------
None



`ps.generate_basis_spectra()` will loop over the input array of grain sizes and compute a set of $\tilde{p}_{\lambda_{\rm em}}(\lambda_{\rm abs})$ for each size (or it can be run for only a single grain size). We suggest running using the default grain sizes, which are stored in `ps.grain_sizes` and include 59 logaritmically-spaced sizes ranging from $3.5481-100.0~{\rm \AA}$. You can also specify your own grain sizes.

**NOTE:** The default size distribution/ionization functions in `ps.generate_spectrum()` expect the default grain sizes, and therefore are not compatible with custom grain sizes. This will be fixed in a future version of `pah_spec`, but in the meantime, a workaround is to utilize the `size_dist_neu` and `size_dist_ion` arguments of `ps.generate_spectrum()` to specify your own size distribution arrays.

`ps.generate_spectrum()` expects the size distributions in the form of $f_{\rm ion}\,dn$ for ions and $(1-f_{\rm ion})\,dn$ for neutrals, where $f_{\rm ion}\in[0,1]$ and the expected units of $dn$ are 1 / H atom (see Eq. 15 of [Draine et al. 2021](https://ui.adsabs.harvard.edu/abs/2021ApJ...917....3D/abstract) for details). [This notebook](https://github.com/brandonshensley/Astrodust/blob/main/notebooks/changing_size_distribution_tutorial.ipynb) contains a helpful example of defining and customizing the size distribution.

To create the basis spectra data files, simply execute the following cell.

In [10]:
ps.generate_basis_spectra(ps.grain_sizes, emission_wavelengths, ion=True)
ps.generate_basis_spectra(ps.grain_sizes, emission_wavelengths, ion=False)

This will create two files, `basis_ion.h5` and `basis_neu.h5`. It takes roughly 3 minutes to compute a full set of basis spectra for a single grain, so the routine will take several hours to run for all grain sizes/both  ionization states.

The data files will contain a 3D array with dimensions of `(len(grain_sizes), 474, len(emission_wavelengths))`. 474 is the number of absorbed photon wavelengths $\lambda_{\rm abs}$ for which we have computed a basis spectrum $\tilde{p}_{\lambda_{\rm em}}(\lambda_{\rm abs})$. Each grain size has its own set of 474 $\tilde{p}_{\lambda_{\rm em}}(\lambda_{\rm abs})$, which are defined at each emission wavelength $\lambda_{\rm em}$.

The data files can be accessed with Python using the `h5py` package.

In [ ]:
import h5py

file_ion = h5py.File("basis_ion.h5", "r")
file_ion["lambda_em"] * u.um  # the 1D array of emission wavelengths
file_ion["lambda_abs"] * u.um  # the 1D array of absorbed photon wavelengths
file_ion["grain_sizes"] * u.AA  # the 1D array of grain sizes
file_ion["basis_spectra"] * u.erg / (u.cm * u.s)  # the 3D basis spectrum array

Once you have successfully created your `basis_ion.h5` and `basis_neu.h5` data files, you can use them to generate PAH spectra by creating a new `PahSpec` object and specifying the `basis_dir` argument

In [ ]:
ps = pah_spec.PahSpec(basis_dir="/path/to/custom/basis/spectra/")

See the installation instructions on [pahspec.readthedocs.io](https://pah-spec.readthedocs.io/en/latest/index.html) for further details on specifying the location of basis spectra data files.